# nomad run analysis — sanity checks and run-vs-reference comparison

Reads a nomad run directory (GeoJSON snapshots `network_state_*.geojson`) and its
stdout log, and compares it with a reference run.

Checks, in order:
1. **Run accounting** from the log: final agent states, event counts, agents with no
   valid route (currently *silently marked Arrived* by the engine — they must not be
   counted as served demand).
2. **Network time series**: vehicles on network, mean congestion, entries per 15 min.
3. **Edge-level agreement** with the reference: daily entries per edge (`n_enter`),
   veh-seconds, scatter + rank correlation, largest discrepancies.
4. **Map** of the edge-level difference.

Snapshot fields are taken as written by the engine; whether `n_enter` is cumulative or
per-interval is *checked* below, not assumed.

In [ ]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path('/data/workspaces/fbellisardi/nomad')
CACHE = BASE / 'analysis_cache'; CACHE.mkdir(exist_ok=True)

RUN_NAME = 'palma_weekday_2022-07-20_car_ltm_seed42_baseline'
RUN_DIR  = next((BASE / 'runs' / RUN_NAME).glob('*_00-00_23-59'))
RUN_LOG  = BASE / 'logs' / f'{RUN_NAME}.o'

REF_NAME = 'gm_weekday_2022-07-20_ltm_dcap10_pretrip_seed42'
REF_DIR  = Path('/data/workspaces/fbellisardi/green_mobility/results/palma_de_mallorca/'
                'weekday_full_car_2022-07-20_ltm_dcap10_pretrip_seed42/2022-07-20_00-00_23-59')
REF_LOG  = None   # reference stdout log not located; log-based checks run on RUN only
RUN_DIR, REF_DIR

## 1. Run accounting (log)

In [ ]:
def parse_log(path):
    if path is None or not Path(path).exists():
        return {}
    txt = Path(path).read_text(errors='replace')
    out = {}
    m = re.search(r'OdMatrixDemand: loaded (\d+) entries \((\d+) agents\)', txt)
    if m: out['od_entries'], out['agents_loaded'] = map(int, m.groups())
    m = re.search(r'(\d+) agents \(([\d.]+)%\) have no valid route', txt)
    out['no_route'] = int(m.group(1)) if m else 0
    m = re.search(r'Event types: (.*)', txt)
    if m: out.update({f'ev_{k}': int(v) for k, v in re.findall(r'(\w+)=(\d+)', m.group(1))})
    m = re.search(r'Final states: (.*)', txt)
    if m: out.update({f'final_{k}': int(v) for k, v in re.findall(r'(\w+)=(\d+)', m.group(1))})
    m = re.findall(r'Simulation complete in ([\d.]+)s', txt)
    if m: out['wall_s'] = float(m[-1])
    m = re.search(r'Maximum resident set size \(kbytes\): (\d+)', Path(str(path).replace('.o', '.e')).read_text()
                  if Path(str(path).replace('.o', '.e')).exists() else '')
    if m: out['peak_rss_gb'] = int(m.group(1)) / 1e6
    return out

acc = parse_log(RUN_LOG)
if acc:
    n = acc['agents_loaded']
    arrived = acc.get('final_Arrived', 0)
    # Agents with no route are flagged Arrived at departure: remove them from "served".
    served = arrived - acc['no_route']
    acc_summary = pd.Series({
        'agents loaded': n,
        'no valid route (silently Arrived)': acc['no_route'],
        'Arrived (engine)': arrived,
        'Arrived, excluding no-route': served,
        'teleported events': acc.get('ev_Teleported', 0),
        'still OnLink at end (stuck)': acc.get('final_OnLink', 0),
        'still Waiting at end': acc.get('final_Waiting', 0),
        'arrival rate (engine)': arrived / n,
        'arrival rate (honest)': served / (n - acc['no_route']),
        'wall time [min]': acc.get('wall_s', np.nan) / 60,
        'peak RSS [GB]': acc.get('peak_rss_gb', np.nan),
    })
    display(acc_summary.to_frame('value'))
else:
    print('log not found:', RUN_LOG)

**Read this table first.** If *no valid route* is non-zero, the engine's arrival rate
overstates served demand. Note that `Arrived` in the engine also includes teleported agents
if the engine marks them so — check `ev_Arrive` vs `final_Arrived` in `acc` below.

In [ ]:
pd.Series(acc).to_frame('raw')

## 2. Load snapshots (cached to parquet)

In [ ]:
FIELDS = ['edge_id', 'count', 'congestion', 'travel_time_s', 'free_flow_time_s',
          'flow_veh_h', 'density_veh_km', 'n_enter', 'n_exit', 'veh_seconds']

def load_snapshots(run_dir, tag):
    cache = CACHE / f'{tag}_snapshots.parquet'
    if cache.exists():
        return pd.read_parquet(cache)
    frames = []
    for p in sorted(Path(run_dir).glob('network_state_*.geojson')):
        t = int(p.stem.split('_')[-1])
        feats = json.loads(p.read_text())['features']
        df = pd.DataFrame([f['properties'] for f in feats])[FIELDS]
        df['t_s'] = t
        frames.append(df)
    out = pd.concat(frames, ignore_index=True)
    out.to_parquet(cache)
    return out

def load_static(run_dir, tag):
    cache = CACHE / f'{tag}_static.parquet'
    import geopandas as gpd
    if cache.exists():
        return gpd.read_parquet(cache)
    g = gpd.read_file(Path(run_dir) / 'static_network.geojson')
    g.to_parquet(cache)
    return g

run = load_snapshots(RUN_DIR, RUN_NAME)
ref = load_snapshots(REF_DIR, REF_NAME)
print(len(run), len(ref), run.t_s.nunique(), ref.t_s.nunique())

### Is `n_enter` cumulative or per-interval?
If cumulative, it is non-decreasing in time for every edge. The rest of the notebook
derives per-interval and daily totals accordingly.

In [ ]:
def is_cumulative(df):
    p = df.pivot_table(index='edge_id', columns='t_s', values='n_enter', fill_value=0)
    return bool((p.diff(axis=1).iloc[:, 1:] >= 0).all().all())

CUM = is_cumulative(run)
print('n_enter cumulative:', CUM, '| ref:', is_cumulative(ref))

def daily_edge(df, cum):
    g = df.groupby('edge_id')
    if cum:
        return pd.DataFrame({'n_enter': g.n_enter.max(), 'veh_seconds': g.veh_seconds.max()})
    return pd.DataFrame({'n_enter': g.n_enter.sum(), 'veh_seconds': g.veh_seconds.sum()})

d_run, d_ref = daily_edge(run, CUM), daily_edge(ref, is_cumulative(ref))

## 3. Network time series

In [ ]:
def series(df):
    g = df.groupby('t_s')
    return pd.DataFrame({'veh_on_network': g['count'].sum(),
                         'mean_congestion_loaded': df[df['count'] > 0].groupby('t_s').congestion.mean(),
                         'tt_ratio_loaded': (df[df['count'] > 0].travel_time_s
                                             / df[df['count'] > 0].free_flow_time_s).groupby(df.t_s).mean()})

s_run, s_ref = series(run), series(ref)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for i, col in enumerate(s_run.columns):
    ax[i].plot(s_ref.index / 3600, s_ref[col], label='reference', lw=2, alpha=.6)
    ax[i].plot(s_run.index / 3600, s_run[col], label='this run', lw=1)
    ax[i].set_title(col); ax[i].set_xlabel('hour')
ax[0].legend(); plt.tight_layout()

## 4. Edge-level agreement with the reference

In [ ]:
from scipy.stats import spearmanr
j = d_run.join(d_ref, lsuffix='_run', rsuffix='_ref', how='outer').fillna(0)
for v in ['n_enter', 'veh_seconds']:
    a, b = j[f'{v}_run'], j[f'{v}_ref']
    rho = spearmanr(a, b).statistic
    print(f'{v:12s} total run={a.sum():.4g} ref={b.sum():.4g} ratio={a.sum()/b.sum():.4f} '
          f'| Spearman={rho:.4f} | identical edges={np.mean(np.isclose(a, b)):.2%} '
          f'| max |diff|={np.abs(a-b).max():.4g}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for i, v in enumerate(['n_enter', 'veh_seconds']):
    ax[i].loglog(j[f'{v}_ref'] + 1, j[f'{v}_run'] + 1, ',', alpha=.4)
    lim = [1, max(j[f'{v}_ref'].max(), j[f'{v}_run'].max()) + 1]
    ax[i].plot(lim, lim, 'k--', lw=.8); ax[i].set_xlabel('reference + 1'); ax[i].set_ylabel('run + 1')
    ax[i].set_title(f'daily {v} per edge')
plt.tight_layout()

**Interpretation guide.** Bit-identical results (identical edges = 100%) are expected only
if the engine code, inputs and seed are identical *and* the parallel event loop is
deterministic. The reference was produced by the `green_mobility` checkout of nomad on
2026-10-13; the merged `main` contains engine commits dated 2026-10-19 (event-loop sync fix,
calendar bucket fix). Small, unstructured differences are therefore plausible; large or
spatially structured ones (e.g. a whole corridor) point to a behavioural change.

In [ ]:
j['diff'] = j.n_enter_run - j.n_enter_ref
j['rel'] = j['diff'] / j.n_enter_ref.clip(lower=50)
j.reindex(j['diff'].abs().sort_values(ascending=False).index).head(15)

## 5. Map of the difference

In [ ]:
static = load_static(RUN_DIR, RUN_NAME)
g = static.merge(j[['n_enter_run', 'n_enter_ref', 'diff']], left_on='edge_id', right_index=True, how='left')
fig, ax = plt.subplots(1, 2, figsize=(16, 8))
g.plot(ax=ax[0], column='n_enter_run', cmap='viridis', linewidth=.4,
       norm=plt.matplotlib.colors.LogNorm(vmin=10, vmax=g.n_enter_run.max()), legend=True)
ax[0].set_title('daily entries (this run)'); ax[0].set_axis_off()
lim = np.nanpercentile(np.abs(g['diff']), 99) or 1
g.plot(ax=ax[1], column='diff', cmap='RdBu_r', vmin=-lim, vmax=lim, linewidth=.5, legend=True)
ax[1].set_title('run − reference (daily entries)'); ax[1].set_axis_off()
plt.tight_layout()